# Uniform Cost Search: Trailer from Nairobi to Kisumu
**Group:** Denley Kipkoech 183787

A trailer has to travel from **Nairobi** to **Kisumu** using the highway network from Part 1. We use Uniform Cost Search (UCS) to find the cheapest route in kilometres.

## 1. Problem formulation
| Element | Definition |
|---|---|
| States | The town the trailer is in |
| Initial state | Nairobi |
| Goal test | Is the town Kisumu? |
| Actions | Drive along a highway to a neighbouring town |
| Path cost | Total distance in km |

## 2. The graph
Distances are from the Part 1 graph (`directed_graph.png`).

![directed graph](directed_graph.png)

Every highway can be driven both ways, so each road is stored in both directions with the same distance.

In [9]:

roads = [
    ("Nairobi", "Thika", 45),
    ("Nairobi", "Naivasha", 90),
    ("Thika", "Murang'a", 40),
    ("Murang'a", "Nyeri", 55),
    ("Nyeri", "Nyahururu", 95),
    ("Nyahururu", "Nakuru", 90),
    ("Naivasha", "Nakuru", 65),
    ("Nakuru", "Kericho", 105),
    ("Nakuru", "Eldoret", 155),
    ("Kericho", "Kisumu", 105),
    ("Eldoret", "Kisumu", 150),
]

graph = {}
for town1, town2, km in roads:
    graph.setdefault(town1, {})
    graph.setdefault(town2, {})
    graph[town1][town2] = km
    graph[town2][town1] = km     

for town in graph:
    print(town, "->", graph[town])

Nairobi -> {'Thika': 45, 'Naivasha': 90}
Thika -> {'Nairobi': 45, "Murang'a": 40}
Naivasha -> {'Nairobi': 90, 'Nakuru': 65}
Murang'a -> {'Thika': 40, 'Nyeri': 55}
Nyeri -> {"Murang'a": 55, 'Nyahururu': 95}
Nyahururu -> {'Nyeri': 95, 'Nakuru': 90}
Nakuru -> {'Nyahururu': 90, 'Naivasha': 65, 'Kericho': 105, 'Eldoret': 155}
Kericho -> {'Nakuru': 105, 'Kisumu': 105}
Eldoret -> {'Nakuru': 155, 'Kisumu': 150}
Kisumu -> {'Kericho': 105, 'Eldoret': 150}


## 3. UCS implementation
How it works (same as in class):
1. The priority queue starts with only Nairobi at cost 0.
2. Each time, we take the town with the **lowest total cost so far** out of the queue.
3. If it is Kisumu, we stop. The goal is checked when a town is *removed* from the queue, not when it is added, because a cheaper way to Kisumu might still be waiting in the queue.
4. If not, we mark the town as explored and add its unexplored neighbours with their new total cost.
5. A town can end up in the queue twice with different costs (e.g. Nyahururu). The costlier copy is skipped when it is removed, because the town is already explored.

In [ ]:
def uniform_cost_search(start, goal):
    # each queue item is (cost so far, town, path taken)
    queue = [(0, start, [start])]
    explored = []
    step = 1

    while len(queue) > 0:
        queue.sort()                      
        cost, town, path = queue.pop(0)

       
        if town in explored:
            print("Skipping old entry for", town, "with cost", cost, "\n")
            continue

        print("Step", step, ": expand", town, "with cost", cost)
        step += 1

        if town == goal:
            print("  We can stop here since we found the goal.\n")
            return cost, path

        explored.append(town)

        for neighbour in graph[town]:
            if neighbour in explored:
                continue
            new_cost = cost + graph[town][neighbour]
            queue.append((new_cost, neighbour, path + [neighbour]))
            print("    add", neighbour, "with cost", new_cost)

        queue.sort()
        print("  queue:", [(c, t) for c, t, p in queue])
        print("  Explored:", explored, "\n")

    return None, None   

## 4. Running the search (step-by-step trace)

In [11]:
best_cost, best_path = uniform_cost_search("Nairobi", "Kisumu")

Step 1 : expand Nairobi with cost 0
    add Thika with cost 45
    add Naivasha with cost 90
  queue: [(45, 'Thika'), (90, 'Naivasha')]
  Explored: ['Nairobi'] 

Step 2 : expand Thika with cost 45
    add Murang'a with cost 85
  queue: [(85, "Murang'a"), (90, 'Naivasha')]
  Explored: ['Nairobi', 'Thika'] 

Step 3 : expand Murang'a with cost 85
    add Nyeri with cost 140
  queue: [(90, 'Naivasha'), (140, 'Nyeri')]
  Explored: ['Nairobi', 'Thika', "Murang'a"] 

Step 4 : expand Naivasha with cost 90
    add Nakuru with cost 155
  queue: [(140, 'Nyeri'), (155, 'Nakuru')]
  Explored: ['Nairobi', 'Thika', "Murang'a", 'Naivasha'] 

Step 5 : expand Nyeri with cost 140
    add Nyahururu with cost 235
  queue: [(155, 'Nakuru'), (235, 'Nyahururu')]
  Explored: ['Nairobi', 'Thika', "Murang'a", 'Naivasha', 'Nyeri'] 

Step 6 : expand Nakuru with cost 155
    add Nyahururu with cost 245
    add Kericho with cost 260
    add Eldoret with cost 310
  queue: [(235, 'Nyahururu'), (245, 'Nyahururu'), (260

## 5. Optimal path and cost

In [12]:
print("Optimal path:", " -> ".join(best_path))
print("Total cost:", best_cost, "km")

# check by adding up the road distances along the path
check = 0
for i in range(len(best_path) - 1):
    check += graph[best_path[i]][best_path[i + 1]]
print("Check:", check, "km")

Optimal path: Nairobi -> Naivasha -> Nakuru -> Kericho -> Kisumu
Total cost: 365 km
Check: 365 km
